**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Estimation Theory

The missing link between [Random Variables](../Analysis/Random_Variables.ipynb) and every filter/model in this curriculum: given noisy data, what is the *best* guess of the underlying parameter — and what does 'best' even mean? Four sessions: maximum likelihood, the Cramér–Rao floor no estimator can beat, Bayesian estimation, and sufficiency.

## 0. Introduction

Setup: data $x_1, \dots, x_n \sim p(x; \theta)$, unknown parameter $\theta$. An *estimator* $\hat{\theta}(x_{1:n})$ is any function of the data — the question is which ones are good, judged by **bias** $E[\hat\theta] - \theta$ and **variance**.

## 1. Pre-requisites

- [Random Variables](../Analysis/Random_Variables.ipynb) — densities, expectation, LOTUS.
- [Optimization](../Optimization/Optimization.ipynb) S1–S2 — we'll maximize likelihoods.
- [Independence](../Analysis/Independence.ipynb) — i.i.d. sampling and the LLN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Maximum Likelihood* (~35 min)
**Goal:** derive MLE for the Gaussian; understand log-likelihood as the natural loss.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (CRLB).

---

## 2. The Maximum Likelihood Principle

💡 **Intuition.** Flip the density around: instead of 'given $\theta$, how probable is data $x$?', ask 'given the data I *saw*, which $\theta$ would have made it least surprising?' The likelihood $L(\theta) = \prod_i p(x_i; \theta)$ scores each candidate; MLE picks the top. Taking logs turns the product into a sum (i.i.d.!) without moving the argmax — and that log-likelihood sum is where nearly every ML loss function comes from: least squares *is* Gaussian MLE, cross-entropy *is* categorical MLE.

### Derivation: Gaussian mean and variance

For $x_i \sim \mathcal{N}(\mu, \sigma^2)$:
$$\log L = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (x_i - \mu)^2$$
$\partial_\mu \log L = \frac{1}{\sigma^2}\sum_i (x_i - \mu) = 0 \Rightarrow \hat\mu = \bar{x}$ — the sample mean, and note: maximizing the Gaussian likelihood in $\mu$ *is* minimizing squared error.

$\partial_{\sigma^2} \log L = 0 \Rightarrow \hat{\sigma}^2 = \frac{1}{n}\sum_i (x_i - \bar{x})^2$ — biased by the factor $\frac{n-1}{n}$ (it 'spends' one data point estimating $\mu$ first). MLE is not automatically unbiased; it *is* consistent and asymptotically optimal.

In [ ]:
# Likelihood surface for a small Gaussian sample — the argmax IS the estimate

# YOUR CODE HERE


---
### 🕐 Session 2 of 4 — *Bias, Variance & the Cramér–Rao Bound* (~40 min)
**Goal:** prove the variance floor; check estimators against it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Bayes).

---

## 3. The Cramér–Rao Lower Bound

💡 **Intuition.** How well *could* any unbiased estimator possibly do? It depends on how much the data's distribution *moves* when $\theta$ moves. If wiggling $\theta$ barely changes $p(x;\theta)$, the data barely carries information about $\theta$, and no cleverness can recover it. **Fisher information** $I(\theta)$ quantifies that sensitivity, and CRLB says: variance $\ge 1/(n I(\theta))$. It's the thermodynamic limit of estimation — the reference line every tracking paper plots.

**Definition.** $I(\theta) = E\big[ (\partial_\theta \log p(x;\theta))^2 \big]$ (the *score*'s variance; the score has mean zero).

**Theorem (CRLB).** For unbiased $\hat\theta$ from $n$ i.i.d. samples (regularity assumed): $\mathrm{Var}(\hat\theta) \ge \frac{1}{n I(\theta)}$.

**Proof sketch.** Unbiasedness $\int \hat\theta \, p \, dx = \theta$; differentiate both sides in $\theta$ to get $\mathrm{Cov}(\hat\theta, s) = 1$ where $s$ is the total score. Cauchy–Schwarz ([Measure Theory](../Analysis/Measure_Theory.ipynb)'s Hölder with $p = q = 2$) then gives $1 \le \mathrm{Var}(\hat\theta) \, \mathrm{Var}(s) = \mathrm{Var}(\hat\theta) \, n I(\theta)$. $\blacksquare$

**Example.** Gaussian mean: $\log p = -(x-\mu)^2/2\sigma^2 + c$, score $= (x-\mu)/\sigma^2$, so $I = 1/\sigma^2$ and CRLB $= \sigma^2/n$ — *exactly* the variance of $\bar{x}$. The sample mean is **efficient**: it sits on the floor.

In [ ]:
# Monte Carlo: three estimators of a Gaussian mean vs the CRLB

# YOUR CODE HERE


---
### 🕐 Session 3 of 4 — *Bayesian Estimation* (~35 min)
**Goal:** treat θ as random; derive MMSE = posterior mean; connect to Kalman.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (sufficiency).

---

## 4. The Bayesian View

💡 **Intuition.** The frequentist asks 'what would happen over repeated experiments?' The Bayesian carries a **belief** about $\theta$ (a prior), and Bayes' rule updates it with data into a posterior — the same belief-update choreography as the [Kalman filter](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb), which is exactly Bayesian estimation for linear-Gaussian models, run recursively.

**MMSE estimator.** Minimizing $E[(\hat\theta - \theta)^2]$ over all functions of the data gives $\hat\theta_{MMSE} = E[\theta \mid x]$ — the posterior mean. (Proof: expand the square around the conditional mean; the cross term vanishes by iterated expectation.)

**Worked example — Gaussian prior + Gaussian data.** Prior $\theta \sim \mathcal{N}(\mu_0, \tau^2)$, data $x_i \sim \mathcal{N}(\theta, \sigma^2)$. The posterior is Gaussian with
$$E[\theta \mid x] = \frac{\tau^2}{\tau^2 + \sigma^2/n} \, \bar{x} \;+\; \frac{\sigma^2/n}{\tau^2 + \sigma^2/n} \, \mu_0$$
— a **precision-weighted blend** of data and prior. Compare the Kalman gain: same formula, same trust dial. Little data ⇒ lean on the prior; lots of data ⇒ the prior washes out and MMSE → MLE.

In [ ]:
# Watch the posterior sharpen and slide from prior to truth as data arrives

# YOUR CODE HERE


---
### 🕐 Session 4 of 4 — *Sufficiency* (~30 min)
**Goal:** find the statistics that capture ALL the information; compress data without losing θ.
**Builds on:** Sessions 1–3.

---

## 5. Sufficient Statistics

💡 **Intuition.** Sometimes a summary of the data is *lossless for the parameter*: once you know the sample mean of Gaussian data, the individual samples carry no further information about $\mu$. That summary is a **sufficient statistic** — the ultimate lossy compression with zero loss *for the question asked*. It's why the Kalman filter can carry just a mean and covariance instead of all past data.

**Definition.** $T(x)$ is sufficient for $\theta$ if the conditional distribution of the data given $T$ does not depend on $\theta$.

**Fisher–Neyman factorization.** $T$ is sufficient iff $p(x; \theta) = g(T(x), \theta) \, h(x)$ — the density splits into a part that sees $\theta$ only through $T$, and a $\theta$-free part.

**Example (Gaussian, known σ).** $\prod_i p(x_i;\mu) \propto \exp\big( \frac{\mu}{\sigma^2} \sum_i x_i - \frac{n\mu^2}{2\sigma^2}\big) \cdot h(x)$ — $\theta$ meets the data only through $\sum_i x_i$. The sample sum (equivalently mean) is sufficient. **Rao–Blackwell** (stated): conditioning any unbiased estimator on a sufficient statistic never increases variance — good estimators should *only* depend on $T$.

In [ ]:
# Empirical check of Rao–Blackwell's spirit: an estimator ignoring T is beatable
# "First sample only" is unbiased for μ but wasteful; the mean (a function of T) dominates it

# YOUR CODE HERE


## 6. Conclusion

MLE turns densities into losses; CRLB is the floor and Fisher information the currency; Bayes blends prior and data by precision; sufficiency tells you what to keep. You can now *read* the estimation claims in every filtering and ML paper.

---
## Where next

- [Adaptive Filtering: Kalman](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — recursive MMSE with Gaussian everything.
- [Statistical Signal Processing](../../Intro_DSP/Statistical_Signal_Processing.ipynb) — detection: estimation's sibling.
- [Uncertainty in ML](../../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — what happens to these guarantees under deep models.